In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [ ]:
import os

# Kaggle dataset path
BASE_PATH = "/kaggle/input/vinbigdata-chest-xray-abnormalities-detection"
TRAIN_DICOM_PATH = os.path.join(BASE_PATH, "train")
CSV_PATH = os.path.join(BASE_PATH, "train.csv")

# Working directory
WORK_PATH = "/kaggle/working/CliniScan"
IMAGE_SAVE_PATH = os.path.join(WORK_PATH, "images")
LABEL_SAVE_PATH = os.path.join(WORK_PATH, "labels")

os.makedirs(IMAGE_SAVE_PATH, exist_ok=True)
os.makedirs(LABEL_SAVE_PATH, exist_ok=True)

print("Working directory created.")

## Loading the input Datasets

In [1]:
import os

print(os.listdir("/kaggle/input"))

['vinbigdata-chest-xray-abnormalities-detection', 'notebooks']


In [4]:
import os
print(os.listdir("/kaggle/input/notebooks"))

['ketkirahegaonkar']


In [5]:
print(os.listdir("/kaggle/input/notebooks/ketkirahegaonkar"))

['x-ray-draft']


In [7]:
print(os.listdir("/kaggle/input/notebooks/ketkirahegaonkar/x-ray-draft"))

['png_images', '__results__.html', 'png_images_backup.zip', '__notebook__.ipynb', '__output__.json', 'custom.css']


In [8]:
IMAGE_DIR = "/kaggle/input/notebooks/ketkirahegaonkar/x-ray-draft/png_images"

print("Total images:", len(os.listdir(IMAGE_DIR)))

Total images: 7000


# Load CSV

In [10]:
import pandas as pd

csv_path = "/kaggle/input/vinbigdata-chest-xray-abnormalities-detection/train.csv"

df = pd.read_csv(csv_path)

print(df.head())
print("Total annotations:", len(df))

                           image_id          class_name  class_id rad_id  \
0  50a418190bc3fb1ef1633bf9678929b3          No finding        14    R11   
1  21a10246a5ec7af151081d0cd6d65dc9          No finding        14     R7   
2  9a5094b2563a1ef3ff50dc5c7ff71345        Cardiomegaly         3    R10   
3  051132a778e61a86eb147c7c6f564dfe  Aortic enlargement         0    R10   
4  063319de25ce7edb9b1c6b8881290140          No finding        14    R10   

    x_min   y_min   x_max   y_max  
0     NaN     NaN     NaN     NaN  
1     NaN     NaN     NaN     NaN  
2   691.0  1375.0  1653.0  1831.0  
3  1264.0   743.0  1611.0  1019.0  
4     NaN     NaN     NaN     NaN  
Total annotations: 67914


In [11]:
print("Unique class IDs:")
print(df["class_id"].unique())

Unique class IDs:
[14  3  0 11  5  8 13  7  1  9  6 10  2  4 12]


# Remove "No finding"

In [14]:
df = df[df["class_id"] !=14]

print("After removing No Finding:")
print("Total annotations:", len(df))

After removing No Finding:
Total annotations: 36096


In [15]:
import os

IMAGE_DIR = "/kaggle/input/notebooks/ketkirahegaonkar/x-ray-draft/png_images"

available_images = set([f.replace(".png", "") for f in os.listdir(IMAGE_DIR)])

df = df[df["image_id"].isin(available_images)]

print("Annotations after matching PNG images:", len(df))

Annotations after matching PNG images: 17120


In [16]:
import os

LABEL_DIR = "/kaggle/working/yolo_labels"
os.makedirs(LABEL_DIR, exist_ok=True)

print("Label folder created.")

Label folder created.


# CSV TO YOLO FORMAT

In [17]:
from tqdm import tqdm

IMG_SIZE = 1024

for image_id, group in tqdm(df.groupby("image_id")):
    
    label_path = os.path.join(LABEL_DIR, image_id + ".txt")
    
    with open(label_path, "w") as f:
        
        for _, row in group.iterrows():
            
            class_id = int(row["class_id"])
            
            x_min = row["x_min"]
            y_min = row["y_min"]
            x_max = row["x_max"]
            y_max = row["y_max"]
            
            # Convert to YOLO format
            x_center = ((x_min + x_max) / 2) / IMG_SIZE
            y_center = ((y_min + y_max) / 2) / IMG_SIZE
            width = (x_max - x_min) / IMG_SIZE
            height = (y_max - y_min) / IMG_SIZE
            
            f.write(f"{class_id} {x_center} {y_center} {width} {height}\n")

print("YOLO labels created successfully!")

print("CSV → YOLO conversion completed.")

100%|██████████| 2043/2043 [00:01<00:00, 1702.25it/s]

YOLO labels created successfully!
CSV → YOLO conversion completed.


# VERIFYING YOLO LABELS

In [18]:
print("Total label files:", len(os.listdir(LABEL_DIR)))

Total label files: 2043


In [21]:
for root, dirs, files in os.walk("/kaggle/working"):
    print("Folder:", root)

Folder: /kaggle/working
Folder: /kaggle/working/yolo_labels
Folder: /kaggle/working/.virtual_documents


# CREATING PROPER YOLO STRUCTURE

In [24]:
import os

print(os.listdir("/kaggle/input/notebooks/ketkirahegaonkar/x-ray-draft"))

['png_images', '__results__.html', 'png_images_backup.zip', '__notebook__.ipynb', '__output__.json', 'custom.css']


In [25]:
print(len(os.listdir("/kaggle/input/notebooks/ketkirahegaonkar/x-ray-draft/png_images")))

7000


In [26]:
import os

BASE_DIR = "/kaggle/working/CliniScan"
IMAGE_DIR = os.path.join(BASE_DIR, "images")
LABEL_DIR = os.path.join(BASE_DIR, "labels")

os.makedirs(IMAGE_DIR, exist_ok=True)
os.makedirs(LABEL_DIR, exist_ok=True)

print("Folders created successfully ✅")

Folders created successfully ✅


In [27]:
import shutil

SOURCE_IMAGE_DIR = "/kaggle/input/notebooks/ketkirahegaonkar/x-ray-draft/png_images"

for file in os.listdir(SOURCE_IMAGE_DIR):
    if file.endswith(".png"):
        shutil.copy(
            os.path.join(SOURCE_IMAGE_DIR, file),
            os.path.join(IMAGE_DIR, file)
        )

print("Images copied successfully ✅")

Images copied successfully ✅


In [28]:
SOURCE_LABEL_DIR = "/kaggle/working/yolo_labels"

for file in os.listdir(SOURCE_LABEL_DIR):
    if file.endswith(".txt"):
        shutil.copy(
            os.path.join(SOURCE_LABEL_DIR, file),
            os.path.join(LABEL_DIR, file)
        )

print("Labels copied successfully ✅")

Labels copied successfully ✅


In [29]:
print("Total Images:", len(os.listdir(IMAGE_DIR)))
print("Total Labels:", len(os.listdir(LABEL_DIR)))

Total Images: 7000
Total Labels: 2043


In [30]:
!du -sh /kaggle/working/yolo_labels


8.2M	/kaggle/working/yolo_labels


In [31]:
!du -sh /kaggle/working/CliniScan


3.2G	/kaggle/working/CliniScan


In [32]:
import os

for root, dirs, files in os.walk("/kaggle/working/CliniScan"):
    print("Folder:", root)
    print("Subfolders:", dirs)
    print("Files:", files[:5])  # show first 5 files only
    print("-"*50)

Folder: /kaggle/working/CliniScan
Subfolders: ['images', 'labels']
Files: []
--------------------------------------------------
Folder: /kaggle/working/CliniScan/images
Subfolders: []
Files: ['b08e7e6fddc01f31892e0b6ab73fd847.png', '15231685e2711bfe8eaef46418877d94.png', '8b4d5c40cadb0b8b8c10f01c0deadad9.png', 'b8c2817b6550ea85ec7872b491fed71e.png', '1730d2ebd1cc96e9e5656cdf916ac7f8.png']
--------------------------------------------------
Folder: /kaggle/working/CliniScan/labels
Subfolders: []
Files: ['36a0490889068162384a000b02d37ad4.txt', '7e705e1d7561ee00863609c72d39aded.txt', '708249b5f1cff1ae05f850da5fc2b625.txt', '38791e51fbea01ae3935f311e8878f1c.txt', 'abac6c7bdaf2149035157fd02883dbcd.txt']
--------------------------------------------------


In [33]:
!find /kaggle/working/CliniScan -maxdepth 3 -type d

/kaggle/working/CliniScan
/kaggle/working/CliniScan/images
/kaggle/working/CliniScan/labels


# CREATE TRAIN AND VAL FOLDERS

In [34]:
import os

base_path = "/kaggle/working/CliniScan"

folders = [
    "images/train",
    "images/val",
    "labels/train",
    "labels/val"
]

for folder in folders:
    os.makedirs(os.path.join(base_path, folder), exist_ok=True)

print("Folders created successfully!")

Folders created successfully!


In [35]:
!find /kaggle/working/CliniScan -maxdepth 3 -type d

/kaggle/working/CliniScan
/kaggle/working/CliniScan/images
/kaggle/working/CliniScan/images/train
/kaggle/working/CliniScan/images/val
/kaggle/working/CliniScan/labels
/kaggle/working/CliniScan/labels/train
/kaggle/working/CliniScan/labels/val


In [36]:
import os

print("Images folder contains:")
print(os.listdir("/kaggle/working/CliniScan/images")[:10])

Images folder contains:
['b08e7e6fddc01f31892e0b6ab73fd847.png', '15231685e2711bfe8eaef46418877d94.png', '8b4d5c40cadb0b8b8c10f01c0deadad9.png', 'b8c2817b6550ea85ec7872b491fed71e.png', '1730d2ebd1cc96e9e5656cdf916ac7f8.png', '4db952de1d3c2a13cc9d7bad1a7f1570.png', 'd10c5ea4f82c32708ac41f11ace41dcb.png', 'eb01ae94da401eb3ae41845119d40f67.png', '1146548e94da58c6989a1a6408dfc886.png', '6d47aabfec517967b9b34cb52a4d5064.png']


In [37]:
import os
import random
import shutil

base_path = "/kaggle/working/CliniScan"

images_path = os.path.join(base_path, "images")
labels_path = os.path.join(base_path, "labels")

train_img_path = os.path.join(images_path, "train")
val_img_path = os.path.join(images_path, "val")

train_lbl_path = os.path.join(labels_path, "train")
val_lbl_path = os.path.join(labels_path, "val")

# Get all PNG images
all_images = [f for f in os.listdir(images_path) if f.endswith(".png")]

# Shuffle images randomly
random.shuffle(all_images)

# Split 80% train, 20% val
split_index = int(0.8 * len(all_images))
train_images = all_images[:split_index]
val_images = all_images[split_index:]

# Move images + labels
for img in train_images:
    shutil.move(os.path.join(images_path, img), os.path.join(train_img_path, img))
    
    label_file = img.replace(".png", ".txt")
    if os.path.exists(os.path.join(labels_path, label_file)):
        shutil.move(os.path.join(labels_path, label_file), os.path.join(train_lbl_path, label_file))

for img in val_images:
    shutil.move(os.path.join(images_path, img), os.path.join(val_img_path, img))
    
    label_file = img.replace(".png", ".txt")
    if os.path.exists(os.path.join(labels_path, label_file)):
        shutil.move(os.path.join(labels_path, label_file), os.path.join(val_lbl_path, label_file))

print("Dataset split completed!")
print("Train images:", len(train_images))
print("Validation images:", len(val_images))

Dataset split completed!
Train images: 5600
Validation images: 1400


In [38]:
import os

print("Train Images:", len(os.listdir("/kaggle/working/CliniScan/images/train")))
print("Train Labels:", len(os.listdir("/kaggle/working/CliniScan/labels/train")))

print("Val Images:", len(os.listdir("/kaggle/working/CliniScan/images/val")))
print("Val Labels:", len(os.listdir("/kaggle/working/CliniScan/labels/val")))

Train Images: 5600
Train Labels: 1625
Val Images: 1400
Val Labels: 418


 # CREATING DATA.YAML

In [39]:
yaml_content = """
path: /kaggle/working/CliniScan
train: images/train
val: images/val

nc: 14  # change if your number of classes is different

names: [
    "Aortic enlargement",
    "Atelectasis",
    "Calcification",
    "Cardiomegaly",
    "Consolidation",
    "ILD",
    "Infiltration",
    "Lung Opacity",
    "Nodule/Mass",
    "Other lesion",
    "Pleural effusion",
    "Pleural thickening",
    "Pneumothorax",
    "Pulmonary fibrosis"
]
"""

with open("/kaggle/working/CliniScan/data.yaml", "w") as f:
    f.write(yaml_content)

print("data.yaml created successfully!")

data.yaml created successfully!


In [40]:
print(os.listdir("/kaggle/working/CliniScan"))

['images', 'data.yaml', 'labels']


In [ ]:
!zip -r CliniScan_Milestone1_Final.zip /kaggle/working/CliniScan

  adding: kaggle/working/CliniScan/ (stored 0%)
  adding: kaggle/working/CliniScan/images/ (stored 0%)
  adding: kaggle/working/CliniScan/images/train/ (stored 0%)
  adding: kaggle/working/CliniScan/images/train/b08e7e6fddc01f31892e0b6ab73fd847.png (deflated 0%)
  adding: kaggle/working/CliniScan/images/train/15231685e2711bfe8eaef46418877d94.png (deflated 0%)
  adding: kaggle/working/CliniScan/images/train/8b4d5c40cadb0b8b8c10f01c0deadad9.png (deflated 0%)
  adding: kaggle/working/CliniScan/images/train/1730d2ebd1cc96e9e5656cdf916ac7f8.png (deflated 0%)
  adding: kaggle/working/CliniScan/images/train/4db952de1d3c2a13cc9d7bad1a7f1570.png (deflated 0%)
  adding: kaggle/working/CliniScan/images/train/d10c5ea4f82c32708ac41f11ace41dcb.png (deflated 2%)
  adding: kaggle/working/CliniScan/images/train/eb01ae94da401eb3ae41845119d40f67.png (deflated 1%)
  adding: kaggle/working/CliniScan/images/train/1146548e94da58c6989a1a6408dfc886.png (deflated 2%)
  adding: kaggle/working/CliniScan/images/tr